# GMSI Headline Live Polling

## Integration Guide: Global Macro Sentiment Indices — Raw Headline Feed

This notebook demonstrates how to integrate the **Permutable AI GMSI raw headline feed** into an analytical workflow. Each record is a single news headline attributed to a **country** and classified into a **macro topic**, scored by two independent models:

| Score | What it measures | Range |
|---|---|---|
| `directional_score` | Topic-specific direction (e.g. +1 = inflation rising) | [−1, +1] |
| `semantic_sentiment_score` | General language tone (+1 = upbeat, −1 = alarming) | [−1, +1] |

The two scores often **diverge**: surging inflation reported in alarming language is directionally positive (+) but semantically negative (−). This divergence is itself a signal.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset.

### Endpoint

| Endpoint | Path |
|---|---|
| Live | `GET /v1/headlines/feed/live/gmsi/{country}` |
| Historical | `GET /v1/headlines/feed/historical/gmsi/{country}` |
| Topic taxonomy | `GET /v1/headlines/topics/gmsi` |
| Country list | `GET /v1/countries` |

Key parameters: `topic` (full topic name), `match_type` (`DOMESTIC` / `INTERNATIONAL` / `COMBINED`), `limit`.

**Important:** The API omits `country`, `topic`, and `match_type` from each JSON record — stamp them from the request context when storing.

### Pipeline

1. Install dependencies and configure credentials
2. Define fetch and upsert helpers
3. Optional historical backfill
4. Dry-run a single poll
5. Visualise directional and semantic scores
6. Run the 15-minute polling loop
7. Visualise the accumulated window
8. Derive a rolling directional indicator

In [ ]:
%pip install requests pandas matplotlib seaborn plotly


## Step 1 — Configuration

In [ ]:
import getpass
import os
from pathlib import Path


def load_env_file() -> None:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.append(base / '.env')
        candidates.append(base / 'systematic/global_macro_sentiment/app/live_headline_polling/.env')

    for env_path in candidates:
        if not env_path.exists():
            continue
        with env_path.open() as env_file:
            for line in env_file:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"'))
        return


load_env_file()
API_KEY = os.environ.get('API_KEY') or os.environ.get('PERMUTABLE_API_KEY')
if not API_KEY:
    try:
        API_KEY = getpass.getpass('Enter your Permutable AI API key: ')
    except Exception as exc:
        raise RuntimeError('Set API_KEY in permutable-examples/.env or export API_KEY before running this notebook.') from exc

BASE_URL = 'https://copilot-api.permutable.ai/v1'
HEADERS = {'x-api-key': API_KEY}

# ── Poll configuration ────────────────────────────────────────────────────────
COUNTRIES   = ['US', 'JP', 'IT', 'GB', 'FR', 'DE', 'CA']           # ISO-2 codes (GET /v1/countries)
TOPICS      = ['Monetary Policy-Policy Tools-Interest Rates', 'Economic Data-Inflation-Energy', 'Economic Data-Inflation-Food', 'Fiscal Policy-Fiscal Stance']  # GET /v1/headlines/topics/gmsi
MATCH_TYPE  = 'COMBINED'                          # DOMESTIC | INTERNATIONAL | COMBINED
POLL_INTERVAL_SECONDS = 900                       # 15 minutes
BACKFILL_DAYS = 7
DB_PATH = '/tmp/gmsi_headlines_nb.db'

print(f'Config: {len(COUNTRIES)} countries × {len(TOPICS)} topics × {MATCH_TYPE}')




## Step 2 — Discover taxonomy (optional)

Use the taxonomy helpers to browse valid country codes and topic names for your licence.

In [ ]:
import requests
import pandas as pd

def get_topics() -> pd.DataFrame:
    r = requests.get(f'{BASE_URL}/headlines/topics/gmsi', headers=HEADERS, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()['topics'])[['name', 'description']]

def get_countries(limit: int = 100) -> pd.DataFrame:
    r = requests.get(f'{BASE_URL}/countries', headers=HEADERS, params={'limit': limit}, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()['countries'])[['id', 'name']]

topics_df = get_topics()
print(f'{len(topics_df)} GMSI topics available')
topics_df.head(10)


## Step 3 — Fetch and storage helpers

In [ ]:
import sqlite3
from datetime import datetime, timezone, timedelta, date

# ── SQLite schema ─────────────────────────────────────────────────────────────

def setup_database(db_path: str = DB_PATH) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS gmsi_headlines (
                country                      TEXT NOT NULL,
                topic                        TEXT NOT NULL,
                match_type                   TEXT NOT NULL,
                publication_time             TEXT NOT NULL,
                topic_probability            REAL,
                directional_positive         REAL,
                directional_neutral          REAL,
                directional_negative         REAL,
                directional_score            REAL,
                semantic_sentiment_positive  REAL,
                semantic_sentiment_neutral   REAL,
                semantic_sentiment_negative  REAL,
                semantic_sentiment_score     REAL,
                fetched_at                   TEXT,
                PRIMARY KEY (country, topic, match_type, publication_time)
            )
        ''')
        conn.commit()

setup_database()
print('Database ready:', DB_PATH)

# ── Upsert ────────────────────────────────────────────────────────────────────

COLUMNS = [
    'country', 'topic', 'match_type', 'publication_time', 'topic_probability',
    'directional_positive', 'directional_neutral', 'directional_negative', 'directional_score',
    'semantic_sentiment_positive', 'semantic_sentiment_neutral',
    'semantic_sentiment_negative', 'semantic_sentiment_score', 'fetched_at',
]

def upsert_headlines(records: list, db_path: str = DB_PATH) -> int:
    if not records:
        return 0
    ph = ', '.join(['?'] * len(COLUMNS))
    sql = f"INSERT OR REPLACE INTO gmsi_headlines ({', '.join(COLUMNS)}) VALUES ({ph})"
    rows = [tuple(r.get(c) for c in COLUMNS) for r in records]
    with sqlite3.connect(db_path) as conn:
        conn.executemany(sql, rows)
        conn.commit()
    return len(rows)

# ── Fetch helpers ─────────────────────────────────────────────────────────────

def _stamp(records: list, country: str, topic: str, match_type: str) -> None:
    ts = datetime.now(timezone.utc).isoformat()
    for r in records:
        r['country'] = country
        r['topic']   = topic
        r['match_type'] = match_type
        r['fetched_at'] = ts

def fetch_live(country: str, topic: str, match_type: str = MATCH_TYPE) -> list:
    url = f'{BASE_URL}/headlines/feed/live/gmsi/{country}'
    r = requests.get(url, headers=HEADERS,
                     params={'topic': topic, 'match_type': match_type, 'limit': 1000},
                     timeout=30)
    r.raise_for_status()
    records = r.json()
    _stamp(records, country, topic, match_type)
    return records

def fetch_historical(country: str, topic: str, start_date: date,
                     match_type: str = MATCH_TYPE) -> list:
    url = f'{BASE_URL}/headlines/feed/historical/gmsi/{country}'
    params = {'topic': topic, 'match_type': match_type,
              'start_date': start_date.isoformat(), 'limit': 1000}
    all_records = []
    while True:
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        body = r.json()
        _stamp(body['data'], country, topic, match_type)
        all_records.extend(body['data'])
        if not body['has_more']:
            break
        params['next_token'] = body['next_token']
    return all_records

print('Helpers defined.')


## Step 4 — Historical backfill (optional)

Seed the database with recent history so visualisations have data immediately.
The 7-day backfill is enabled by default so the charts below are populated. Set `RUN_BACKFILL = False` for a very quick dry run.

In [ ]:
RUN_BACKFILL = True   # Set to False for a very quick dry run

if RUN_BACKFILL:
    start = (datetime.utcnow() - timedelta(days=BACKFILL_DAYS)).date()
    for country in COUNTRIES:
        for topic in TOPICS:
            print(f'  Backfilling {country} / {topic}...')
            records = fetch_historical(country, topic, start)
            n = upsert_headlines(records)
            print(f'    {n} rows upserted')
    print('Backfill complete.')
else:
    print('Backfill skipped — set RUN_BACKFILL = True for populated charts.')


## Step 5 — Dry-run: single poll

Fetch one country/topic combination and inspect the response schema.

In [ ]:
sample_records = fetch_live(country=COUNTRIES[0], topic=TOPICS[0])
n = upsert_headlines(sample_records)
print(f'Fetched {len(sample_records)} records for {COUNTRIES[0]} / {TOPICS[0]} | {n} upserted')

if sample_records:
    df_sample = pd.DataFrame(sample_records)
    df_sample['publication_time'] = pd.to_datetime(df_sample['publication_time'], utc=True)
    print(f'Time range: {df_sample["publication_time"].min()} → {df_sample["publication_time"].max()}')
    display(df_sample[[
        'country', 'topic', 'match_type', 'publication_time',
        'topic_probability', 'directional_score', 'semantic_sentiment_score',
    ]].head())
else:
    print('No records returned — the topic may have no recent headlines for this country.')


## Step 6 — Visualise the initial snapshot

Compare directional and semantic scores. The divergence between them often carries signal.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def load_db(db_path: str = DB_PATH, hours: int = 168) -> pd.DataFrame:
    since = (datetime.now(timezone.utc) - timedelta(hours=hours)).isoformat()
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql(
            'SELECT * FROM gmsi_headlines WHERE publication_time >= ? ORDER BY publication_time',
            conn, params=(since,),
        )
    df['publication_time'] = pd.to_datetime(df['publication_time'], utc=True)
    df['directional_score'] = pd.to_numeric(df['directional_score'], errors='coerce')
    df['semantic_sentiment_score'] = pd.to_numeric(df['semantic_sentiment_score'], errors='coerce')
    return df.dropna(subset=['directional_score', 'semantic_sentiment_score'])


df = load_db(hours=168)
print(f'{len(df):,} records loaded from DB')

if df.empty:
    print('No data yet — run the dry-run cell above or the backfill step first.')
else:
    freq = '2h' if (df['publication_time'].max() - df['publication_time'].min()).total_seconds() >= 120 * 3600 else '1h'
    trend = (
        df.set_index('publication_time')
        .resample(freq)
        .agg(
            directional=('directional_score', 'mean'),
            semantic=('semantic_sentiment_score', 'mean'),
            headlines=('directional_score', 'count'),
        )
    )
    trend[['directional', 'semantic']] = trend[['directional', 'semantic']].rolling(3, min_periods=1).mean()

    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=df['publication_time'], y=df['directional_score'], mode='markers',
        marker=dict(size=4, color='#9aa4b2', opacity=0.18),
        name='Headline directional scores', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=trend.index, y=trend['directional'], mode='lines',
        line=dict(color='#58d68d', width=3), name='Directional trend',
    ))
    fig.add_trace(go.Scatter(
        x=trend.index, y=trend['semantic'], mode='lines',
        line=dict(color='#f194c8', width=3, dash='dash'), name='Semantic trend',
    ))
    fig.add_hline(y=0, line_dash='dot', line_color='#666')
    fig.update_layout(
        template='plotly_dark', paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e',
        title=f'GMSI Raw Headlines — Directional vs Semantic ({freq} smoothed trend, last 7d)',
        xaxis_title='Time (UTC)', yaxis_title='Score [-1, +1]', yaxis=dict(range=[-1.05, 1.05]),
        height=420, legend=dict(orientation='h', x=0, y=1.08),
    )
    fig.show()

    counts = df.assign(hour=df['publication_time'].dt.floor('1h')).groupby(['hour', 'country']).size().reset_index(name='headlines')
    fig = go.Figure()
    for country in sorted(counts['country'].unique()):
        sub = counts[counts['country'] == country]
        fig.add_trace(go.Bar(x=sub['hour'], y=sub['headlines'], name=country))
    fig.update_layout(
        template='plotly_dark', paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e',
        title='Headline volume per hour by country', barmode='stack',
        xaxis_title='Time (UTC)', yaxis_title='Headlines', height=320,
        legend=dict(orientation='h', x=0, y=1.1),
    )
    fig.show()




## Step 7 — Divergence scatter

Plot mean directional vs mean semantic per country. Points far from the diagonal have large divergence.

In [ ]:
if not df.empty:
    country_df = (
        df.groupby('country')
        .agg(
            directional=('directional_score', 'mean'),
            semantic=('semantic_sentiment_score', 'mean'),
            headlines=('directional_score', 'count'),
        )
        .sort_values('directional')
        .reset_index()
    )

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Directional vs semantic by country', 'Score distribution'),
        horizontal_spacing=0.14,
    )
    fig.add_trace(go.Bar(
        y=country_df['country'], x=country_df['directional'], orientation='h',
        marker_color='#58d68d', name='Directional',
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        y=country_df['country'], x=country_df['semantic'], orientation='h',
        marker_color='#f194c8', name='Semantic',
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=df['directional_score'], nbinsx=32, marker_color='#58d68d',
        opacity=0.78, name='Directional distribution',
    ), row=1, col=2)
    fig.add_trace(go.Histogram(
        x=df['semantic_sentiment_score'], nbinsx=32, marker_color='#f194c8',
        opacity=0.68, name='Semantic distribution',
    ), row=1, col=2)
    fig.update_layout(
        template='plotly_dark', paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e',
        barmode='overlay', height=430,
        title='Country-level divergence and score distribution',
        legend=dict(orientation='h', x=0, y=1.08),
    )
    fig.update_xaxes(range=[-1.05, 1.05], row=1, col=1)
    fig.update_xaxes(range=[-1.05, 1.05], row=1, col=2)
    fig.show()
else:
    print('No data yet.')




## Step 8 — Live polling loop

Polls all configured country/topic combinations every `POLL_INTERVAL_SECONDS`.
Run this cell and interrupt with the stop button when done.

**Note:** Each country/topic combination is one API request per cycle. Keep the configuration list conservative to stay within your quota.

In [ ]:
import time

RUN_POLLING_LOOP = False   # Set to True to start the live polling loop

if RUN_POLLING_LOOP:
    poll = 0
    while True:
        poll += 1
        t0 = datetime.now(timezone.utc)
        print(f'[Poll {poll}]  {t0.strftime("%H:%M:%S UTC")}')
        total = 0
        for country in COUNTRIES:
            for topic in TOPICS:
                try:
                    records = fetch_live(country, topic)
                    n = upsert_headlines(records)
                    print(f'  {country}/{topic}: {len(records)} fetched | {n} upserted')
                    total += n
                except Exception as exc:
                    print(f'  {country}/{topic}: ERROR — {exc}')
        elapsed = (datetime.now(timezone.utc) - t0).total_seconds()
        wait = max(0, POLL_INTERVAL_SECONDS - elapsed)
        print(f'  Done in {elapsed:.1f}s.  Next poll in {wait/60:.1f} min.\n')
        time.sleep(wait)
else:
    print('Polling loop disabled — set RUN_POLLING_LOOP = True to enable.')


## Step 9 — Accumulated window visualisation

In [ ]:
df = load_db(hours=168)
print(f'{len(df):,} records, {df["country"].nunique()} countries, {df["publication_time"].min()} → {df["publication_time"].max()}')

if not df.empty:
    top_topics = df.groupby('topic').size().nlargest(12).index
    topic_df = (
        df[df['topic'].isin(top_topics)]
        .groupby('topic')
        .agg(directional=('directional_score', 'mean'), headlines=('directional_score', 'count'))
        .sort_values('directional')
        .reset_index()
    )

    fig = go.Figure(go.Bar(
        x=topic_df['directional'], y=topic_df['topic'], orientation='h',
        marker_color=['#e74c3c' if v < 0 else '#2ecc71' for v in topic_df['directional']],
        customdata=topic_df[['headlines']],
        hovertemplate='%{y}<br>Directional: %{x:+.3f}<br>Headlines: %{customdata[0]:,}<extra></extra>',
    ))
    fig.add_vline(x=0, line_dash='dot', line_color='#666')
    fig.update_layout(
        template='plotly_dark', paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e',
        title='Mean directional score by GMSI topic (top by volume)',
        xaxis_title='Mean directional score', xaxis=dict(range=[-1.05, 1.05]),
        height=max(360, 110 + 34 * len(topic_df)),
    )
    fig.show()
else:
    print('No data yet.')




## Step 10 — Rolling directional indicator

Derive a HIGH / NEUTRAL / LOW signal from the 1-hour binned rolling mean of `directional_score`.

In [ ]:
UPPER_THRESHOLD = 0.1
LOWER_THRESHOLD = -0.1
ROLLING_WINDOW = 5   # hours

import plotly.graph_objects as go

if not df.empty:
    df['hour'] = df['publication_time'].dt.floor('1h')
    hourly = (
        df.groupby(['country', 'hour'])
        .agg(
            directional_mean=('directional_score', 'mean'),
            semantic_mean=('semantic_sentiment_score', 'mean'),
            count=('directional_score', 'count'),
        )
        .reset_index()
        .sort_values(['country', 'hour'])
    )
    hourly['directional_smooth'] = hourly.groupby('country')['directional_mean'].transform(
        lambda values: values.rolling(ROLLING_WINDOW, min_periods=1).mean()
    )
    hourly['indicator'] = 'NEUTRAL'
    hourly.loc[hourly['directional_smooth'] >= UPPER_THRESHOLD, 'indicator'] = 'HIGH'
    hourly.loc[hourly['directional_smooth'] <= LOWER_THRESHOLD, 'indicator'] = 'LOW'

    latest = (
        hourly.sort_values('hour')
        .groupby('country')
        .last()
        .reset_index()
        [['country', 'hour', 'directional_smooth', 'semantic_mean', 'count', 'indicator']]
        .sort_values('directional_smooth', ascending=False)
    )
    display(latest)

    pivot = hourly.pivot_table(index='country', columns='hour', values='directional_smooth', aggfunc='mean')
    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        colorscale=[[0, '#c0392b'], [0.5, '#2a2a3e'], [1, '#2ecc71']],
        zmid=0, zmin=-1, zmax=1,
        colorbar=dict(title='Directional'),
        hovertemplate='%{y} · %{x|%b %-d %H:%M}<br>Directional: %{z:+.3f}<extra></extra>',
    ))
    fig.update_layout(
        template='plotly_dark', paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e',
        title=f'{ROLLING_WINDOW}h rolling directional score by country',
        xaxis_title='Time (UTC)', yaxis_title='Country',
        height=max(360, 140 + 44 * len(pivot.index)),
    )
    fig.show()
else:
    print('No data yet.')


